# Configuração do Ambiente e Banco de Dados
Esta célula explica como preparar as pastas e criar o arquivo do banco de dados SQLite para armazenar as tabelas do censo.

In [ ]:
# Importa a biblioteca pandas para manipulação de dados
import pandas as pd
 # Importa sqlite3 para trabalhar com banco de dados SQLite
import sqlite3
 # Importa os para manipulação de diretórios
import os

print("Célula 1: Configurando o ambiente...")  # Mensagem de início

# Definindo os caminhos
caminho_dados = '../dados/'  # Caminho para os arquivos de dados
caminho_bd = '../banco_de_dados/'  # Caminho para o banco de dados

# Cria a pasta para o banco de dados caso não exista
os.makedirs(caminho_bd, exist_ok=True)  # Garante que a pasta do banco exista

# Conectando ao banco SQLite (cria o arquivo fisicamente na pasta)
caminho_sqlite = caminho_bd + 'Banco_Censo_Completo.db'  # Define o caminho do arquivo do banco
conexao = sqlite3.connect(caminho_sqlite)  # Conecta ao banco de dados SQLite

print(f"Banco de dados conectado com sucesso em: {caminho_sqlite}")  # Confirma conexão

Célula 1: Configurando o ambiente...
Banco de dados conectado com sucesso em: ../banco_de_dados/Banco_Censo_Completo.db


# Leitura em Chunks e Envio para o SQLite
Esta célula detalha o processo de leitura dos arquivos CSV em blocos para evitar problemas de memória, padronizando a chave CD_SETOR e salvando os dados no banco SQLite.

In [ ]:
print("Célula 2: Iniciando a importação em blocos (Chunks) para o SQLite...")  # Mensagem de início

# Lista com o nome exato do arquivo e o nome da tabela que será criada no banco
csvs = [
    ('Agregados_por_setores_basico_BR_20250417.csv', 'tb_basico'),  # Arquivo e tabela
    ('Agregados_por_setores_caracteristicas_domicilio1_BR.csv', 'tb_dom1'),
    ('Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.csv', 'tb_dom2'),
    ('Agregados_por_setores_alfabetizacao_BR.csv', 'tb_alfab'),
    ('Agregados_por_setores_cor_ou_raca_BR.csv', 'tb_raca'),
    ('Agregados_por_setores_renda_responsavel_BR.csv', 'tb_renda')
 ]

for csv_file, table_name in csvs:  # Itera sobre os arquivos e tabelas
    print(f"Processando {csv_file} -> Tabela: {table_name}")  # Mensagem de processamento
    
    # Limpa a tabela se você rodar esta célula mais de uma vez
    conexao.execute(f"DROP TABLE IF EXISTS {table_name}")  # Remove tabela anterior
    
    caminho_arquivo = caminho_dados + csv_file  # Caminho do arquivo CSV
    
    # O low_memory=False e dtype=str garantem que os códigos com zeros à esquerda não se percam
    for chunk in pd.read_csv(caminho_arquivo, sep=';', dtype=str, encoding='latin1', chunksize=50000, low_memory=False):  # Lê em blocos
        # Correção Crítica: Padroniza o nome da coluna de ligação ANTES de ir para o banco
        if 'CD_setor' in chunk.columns:
            chunk = chunk.rename(columns={'CD_setor': 'CD_SETOR'})  # Renomeia coluna
        if 'setor' in chunk.columns:
            chunk = chunk.rename(columns={'setor': 'CD_SETOR'})  # Renomeia coluna
        
        # Adiciona o pedaço lido na tabela do SQLite
        chunk.to_sql(table_name, conexao, if_exists='append', index=False)  # Salva chunk no banco

print("\nSucesso! Todos os dados brutos foram guardados no disco (SQLite) sem estourar a RAM.")  # Mensagem de sucesso

Célula 2: Iniciando a importação em blocos (Chunks) para o SQLite...
Processando Agregados_por_setores_basico_BR_20250417.csv -> Tabela: tb_basico
Processando Agregados_por_setores_caracteristicas_domicilio1_BR.csv -> Tabela: tb_dom1
Processando Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.csv -> Tabela: tb_dom2
Processando Agregados_por_setores_alfabetizacao_BR.csv -> Tabela: tb_alfab
Processando Agregados_por_setores_cor_ou_raca_BR.csv -> Tabela: tb_raca
Processando Agregados_por_setores_renda_responsavel_BR.csv -> Tabela: tb_renda

Sucesso! Todos os dados brutos foram guardados no disco (SQLite) sem estourar a RAM.


# Join Completo no SQLite
Esta célula descreve como unir todas as tabelas do banco de dados utilizando comandos SQL, garantindo eficiência e evitando duplicação de colunas.

In [ ]:
print("Célula 3: Executando o JOIN completo dentro do SQLite...")  # Mensagem de início

# Comando SQL para cruzar todas as tabelas
query_join = """
CREATE TABLE base_censo_unificada AS
SELECT *
FROM tb_basico
LEFT JOIN tb_dom1 USING (CD_SETOR)
LEFT JOIN tb_dom2 USING (CD_SETOR)
LEFT JOIN tb_alfab USING (CD_SETOR)
LEFT JOIN tb_raca USING (CD_SETOR)
LEFT JOIN tb_renda USING (CD_SETOR);
"""  # SQL para unir tabelas

# Limpa execuções anteriores para evitar erro de "tabela já existe"
conexao.execute("DROP TABLE IF EXISTS base_censo_unificada")  # Remove tabela anterior

# Roda a união
conexao.execute(query_join)  # Executa o JOIN
conexao.commit()  # Salva alterações

print("Matriz unificada criada com sucesso no banco de dados!")  # Mensagem de sucesso

Célula 3: Executando o JOIN completo dentro do SQLite...
Matriz unificada criada com sucesso no banco de dados!


# Exportação Segura para CSV
Esta célula explica como exportar a base unificada para um arquivo CSV em blocos, protegendo o computador contra problemas de memória durante o processo.

In [ ]:
print("Célula 4: Exportando a base unificada monstruosa para CSV...")  # Mensagem de início

caminho_csv_saida = caminho_bd + 'Base_Censo_Completa_Unificada.csv'  # Caminho do arquivo de saída
query_select = "SELECT * FROM base_censo_unificada"  # SQL para selecionar dados

primeiro_chunk = True  # Controle para cabeçalho

# Puxa 20 mil linhas por vez do banco e escreve no CSV
for chunk in pd.read_sql_query(query_select, conexao, chunksize=20000):  # Lê em blocos
    # mode='a' (append) vai colando os pedaços no final do arquivo
    chunk.to_csv(caminho_csv_saida, mode='a', index=False, sep=';', encoding='utf-8-sig', header=primeiro_chunk)  # Salva chunk no CSV
    
    # O cabeçalho só é escrito no primeiro pedaço
    primeiro_chunk = False  # Desativa cabeçalho para próximos chunks

# Fecha a conexão com o banco para liberar o arquivo
conexao.close()  # Fecha conexão

print(f"Processo perfeito! A base completa foi unificada e exportada para:\n{caminho_csv_saida}")  # Mensagem final

Célula 4: Exportando a base unificada monstruosa para CSV...
Processo perfeito! A base completa foi unificada e exportada para:
../banco_de_dados/Base_Censo_Completa_Unificada.csv
